In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (SparkSession.builder
    .appName("chakhan-price")
    .master("local[*]")
    .getOrCreate())

spark

In [2]:
# 셀 — CSV 로드

In [5]:
PATH = "/home/jovyan/work/행정안전부_착한가격업소 현황_20260630.csv"

df = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "UTF-8")     # 깨지면 UTF-8로 교체
    .option("multiLine", True)
    .option("escape", '"')
    .csv(PATH))

df.printSchema()
print("행 수:", df.count())
print(df.columns)

root
 |-- 시도: string (nullable = true)
 |-- 시군: string (nullable = true)
 |-- 업종: string (nullable = true)
 |-- 업소명: string (nullable = true)
 |-- 연락처: string (nullable = true)
 |-- 주소: string (nullable = true)
 |-- 메뉴1: string (nullable = true)
 |-- 가격1: string (nullable = true)
 |-- 메뉴2: string (nullable = true)
 |-- 가격2: string (nullable = true)
 |-- 메뉴3: string (nullable = true)
 |-- 가격3: string (nullable = true)
 |-- 메뉴4: string (nullable = true)
 |-- 가격4: string (nullable = true)

행 수: 12645
['시도', '시군', '업종', '업소명', '연락처', '주소', '메뉴1', '가격1', '메뉴2', '가격2', '메뉴3', '가격3', '메뉴4', '가격4']


In [6]:
# 셀 3 — 컬럼명 영문화 + 가격 타입 통일

In [7]:
rename_map = {
    "시도": "sido", "시군": "sigun", "업종": "category",
    "업소명": "name", "연락처": "tel", "주소": "address",
    "메뉴1": "menu1", "가격1": "price1",
    "메뉴2": "menu2", "가격2": "price2",
    "메뉴3": "menu3", "가격3": "price3",
    "메뉴4": "menu4", "가격4": "price4",
}
for old, new in rename_map.items():
    df = df.withColumnRenamed(old, new)

# ★ 가격 컬럼 타입 통일 (inferSchema가 컬럼마다 int/string 다르게 잡았을 수 있음)
for c in ["price1", "price2", "price3", "price4"]:
    df = df.withColumn(c, F.col(c).cast("string"))

df.limit(5).toPandas()

,sido,sigun,category,name,tel,address,menu1,price1,menu2,price2,menu3,price3,menu4,price4
0,서울특별시,종로구,양식,돈까스보라,02-741-3455,서울특별시 종로구 대학로5길 5 (연건동),수제 돈까스,7000,돈까스 정식,8000,None,None,None,None
1,서울특별시,종로구,한식,삼삼 뚝배기,02-765-4683,서울특별시 종로구 동숭길 51 (동숭동),된장뚝배기,7500,김치찌개,7500,None,None,None,None
2,서울특별시,종로구,기타요식업,약속커피숍,02-764-1100,서울특별시 종로구 종로 302 (창신동),맥심커피,2000,녹차,2000,None,None,None,None
3,서울특별시,종로구,미용업,영미용실,02-743-6052,서울특별시 종로구 지봉로 62 (숭인동),커트,8000,파마,20000,None,None,None,None
4,서울특별시,종로구,기타요식업,왕관커피숍,02-2266-8365,서울특별시 종로구 종로 222 (종로5가),커피,2000,국산차,2000,None,None,None,None


In [8]:
# 셀 4 — Wide → Long 변환 (핵심)

In [9]:
long_df = (df.select(
        "sido", "sigun", "category", "name", "address",
        F.expr("""
            stack(4,
                1, menu1, price1,
                2, menu2, price2,
                3, menu3, price3,
                4, menu4, price4
            ) as (menu_no, menu, price_raw)
        """)
    )
    .filter(F.col("menu").isNotNull() & (F.trim(F.col("menu")) != ""))
    # 1) 첫 번째 숫자 덩어리만 추출 (콤마 포함 허용)
    .withColumn("price_str",
        F.regexp_extract(F.col("price_raw"), r"(\d{1,3}(?:,\d{3})+|\d+)", 1))
    # 2) 콤마 제거 후 try_cast → 실패해도 NULL, 예외 안 남
    .withColumn("price",
        F.expr("try_cast(regexp_replace(price_str, ',', '') as bigint)"))
    .filter(F.col("price").isNotNull() & (F.col("price") > 0))
    # 3) 상식 밖 값 차단 (10원 미만 / 100만원 초과)
    .filter((F.col("price") >= 500) & (F.col("price") <= 1000000))
    .drop("price_str")
)

long_df.cache()
print("메뉴 단위 행 수:", long_df.count())
long_df.limit(10).toPandas()

메뉴 단위 행 수: 26762


,sido,sigun,category,name,address,menu_no,menu,price_raw,price
0,서울특별시,종로구,양식,돈까스보라,서울특별시 종로구 대학로5길 5 (연건동),1,수제 돈까스,7000,7000
1,서울특별시,종로구,양식,돈까스보라,서울특별시 종로구 대학로5길 5 (연건동),2,돈까스 정식,8000,8000
2,서울특별시,종로구,한식,삼삼 뚝배기,서울특별시 종로구 동숭길 51 (동숭동),1,된장뚝배기,7500,7500
3,서울특별시,종로구,한식,삼삼 뚝배기,서울특별시 종로구 동숭길 51 (동숭동),2,김치찌개,7500,7500
4,서울특별시,종로구,기타요식업,약속커피숍,서울특별시 종로구 종로 302 (창신동),1,맥심커피,2000,2000
5,서울특별시,종로구,기타요식업,약속커피숍,서울특별시 종로구 종로 302 (창신동),2,녹차,2000,2000
6,서울특별시,종로구,미용업,영미용실,서울특별시 종로구 지봉로 62 (숭인동),1,커트,8000,8000
7,서울특별시,종로구,미용업,영미용실,서울특별시 종로구 지봉로 62 (숭인동),2,파마,20000,20000
8,서울특별시,종로구,기타요식업,왕관커피숍,서울특별시 종로구 종로 222 (종로5가),1,커피,2000,2000
9,서울특별시,종로구,기타요식업,왕관커피숍,서울특별시 종로구 종로 222 (종로5가),2,국산차,2000,2000


In [10]:
UNIT_PAT = r"\d+\s*(?:g|kg|ml|리터|인분|장|매|시간)"

clean = (
    long_df
    .withColumn("p_norm", F.regexp_replace(F.col("price_raw"), UNIT_PAT, " "))
    .withColumn("p_first",
        F.regexp_extract(F.col("p_norm"), r"(\d{1,3}(?:,\d{3})+|\d+)", 1))
    .withColumn("price",
        F.expr("try_cast(regexp_replace(p_first, ',', '') as bigint)"))
    .withColumn("is_multi",
        F.col("price_raw").rlike(r"\d\s*[~\-/]\s*\d|\d[^0-9]*\d{3,}"))
    .withColumn("flag",
        F.when(F.col("price").isNull(), "파싱실패")
         .when(F.col("price") < 100, "값이상")
         .otherwise("정상"))
)

clean.cache()
clean.groupBy("flag").count().toPandas()

,flag,count
0,정상,26762


In [11]:
raw_cnt = (df.select(F.expr("stack(4, 1,menu1,price1, 2,menu2,price2, 3,menu3,price3, 4,menu4,price4) as (n,m,p)"))
             .filter(F.col("m").isNotNull() & (F.trim(F.col("m")) != "")).count())
kept = long_df.count()
print(f"원본 {raw_cnt} → 유효 {kept} ({kept/raw_cnt*100:.1f}%)")

원본 26818 → 유효 26762 (99.8%)


In [12]:
# 셀 5 — 집계 3종

In [13]:
# 1) 시도별 업소 수
(df.groupBy("sido").count()
   .orderBy(F.desc("count")).toPandas())

,sido,count
0,서울특별시,2082
1,경기도,1894
2,부산광역시,1204
3,경상남도,888
4,경상북도,795
5,강원특별자치도,739
6,전라남도,662
7,대구광역시,596
8,충청남도,593
9,대전광역시,562


In [14]:
# 2) 업종별 가격 통계 (메뉴 단위)
(long_df.groupBy("category")
   .agg(
       F.count("*").alias("menu_cnt"),
       F.countDistinct("name").alias("shop_cnt"),
       F.round(F.avg("price"), 0).cast("int").alias("avg_price"),
       F.expr("percentile_approx(price, 0.5)").alias("median_price"),
       F.min("price").alias("min_price"),
       F.max("price").alias("max_price"),
   )
   .orderBy(F.desc("menu_cnt")).toPandas())

,category,menu_cnt,shop_cnt,avg_price,median_price,min_price,max_price
0,한식,15650,6892,10522,8000,500,176000
1,미용업,4169,1626,19288,15000,2000,700000
2,기타요식업,2125,985,3819,3000,500,140000
3,중식,1541,621,7418,6500,1800,89000
4,기타비요식업,818,413,68816,20000,500,1000000
5,이용업,645,306,11490,10000,1200,40000
6,세탁업,443,208,5979,5000,1200,50000
7,일식,435,211,15969,10000,1500,130000
8,양식,416,195,9550,9000,1000,31000
9,숙박업,319,145,64288,50000,10000,300000


In [15]:
# 3) 전국 최다 등장 메뉴 TOP 20 + 지역별 가격차
(long_df.groupBy("menu")
   .agg(
       F.count("*").alias("cnt"),
       F.round(F.avg("price"), 0).cast("int").alias("avg_price"),
       F.stddev("price").alias("std_price"),
   )
   .filter(F.col("cnt") >= 30)
   .orderBy(F.desc("cnt")).limit(20).toPandas())

,menu,cnt,avg_price,std_price
0,커트,1129,9847,2900.931353
1,염색,633,22114,9781.101390
2,김치찌개,632,7538,1269.310856
3,파마,604,28406,8035.330449
4,칼국수,565,6287,1166.236581
5,아메리카노,481,2513,622.026398
6,된장찌개,476,7194,1203.743883
7,짜장면,455,5402,1091.001362
8,짬뽕,428,6929,1168.362141
9,잔치국수,324,5440,942.124470


In [16]:
# 셀 6 — 윈도우 함수

In [17]:
from pyspark.sql.window import Window

# 짜장면 기준 시도별 최저가 업소
target = "짜장면"
w = Window.partitionBy("sido").orderBy(F.asc("price"))

(long_df
   .filter(F.col("menu").contains(target))
   .withColumn("rn", F.row_number().over(w))
   .filter(F.col("rn") == 1)
   .select("sido", "sigun", "name", "menu", "price")
   .orderBy("price").toPandas())

,sido,sigun,name,menu,price
0,대구광역시,중구,만리장성,짜장면,2000
1,서울특별시,금천구,왕래성,짜장면,2000
2,울산광역시,동구,북경,짜장면,2000
3,인천광역시,강화군,중화요리금성,짜장면,2000
4,부산광역시,부산진구,옛날짜장,짜장면,2500
5,경기도,파주시,사천성,짜장면(매장식사),3000
6,경상남도,창원시,악어반점,옛날짜장면,3000
7,경상북도,경주시,구황반점,짜장면(소),3000
8,광주광역시,북구,원보,짜장면,3000
9,세종특별자치시,None,광진손짜장,짜장면,3000


In [18]:
# 업종 평균 대비 저렴한 정도 (같은 업종 내 백분위)
w2 = Window.partitionBy("category")

(long_df
   .withColumn("cat_avg", F.round(F.avg("price").over(w2), 0))
   .withColumn("gap_pct", F.round((F.col("price") / F.col("cat_avg") - 1) * 100, 1))
   .filter(F.col("category").isNotNull())
   .filter(F.col("category") != "기타비요식업")
   .select("sido", "name", "category", "menu", "price", "cat_avg", "gap_pct")
   .orderBy("gap_pct").limit(20).toPandas())

,sido,name,category,menu,price,cat_avg,gap_pct
0,서울특별시,월곡동할매떡볶이,한식,김밥 및 계란,500,10522.0,-95.2
1,서울특별시,따뜻한밥상 홍제,한식,계란후라이,500,10522.0,-95.2
2,부산광역시,광화식품,한식,순두부 400g,500,10522.0,-95.2
3,부산광역시,광화식품,한식,연두부 400g,500,10522.0,-95.2
4,부산광역시,명성식당,한식,어묵,500,10522.0,-95.2
5,대전광역시,대박손만두분식,한식,찐빵(1개),500,10522.0,-95.2
6,충청남도,원북수제먹거리,한식,꽈배기,500,10522.0,-95.2
7,경상북도,슈퍼떡방앗간,한식,고춧가루 수공비(600g),500,10522.0,-95.2
8,서울특별시,월곡동할매떡볶이,한식,튀김,700,10522.0,-93.3
9,서울특별시,털보네떡꼬치,한식,어묵(1개),700,10522.0,-93.3


In [19]:
# 셀 7 — Parquet 저장

In [20]:
(long_df.write.mode("overwrite")
   .partitionBy("sido")
   .parquet("/home/jovyan/work/output/shops_long2"))

!ls /home/jovyan/work/99spark/output/shops_long

ls: cannot access '/home/jovyan/work/99spark/output/shops_long': No such file or directory


In [21]:
# CSV로도 저장해서 용량 비교
long_df.write.mode("overwrite").csv("/home/jovyan/work/output/shops_csv2")

!du -sh /home/jovyan/work/99spark/output/shops_long
!du -sh /home/jovyan/work/99spark/output/shops_csv

du: cannot access '/home/jovyan/work/99spark/output/shops_long': No such file or directory
du: cannot access '/home/jovyan/work/99spark/output/shops_csv': No such file or directory


In [22]:
back = spark.read.parquet("/home/jovyan/work/output/shops_long2")
back.filter(F.col("sido") == "서울특별시").count()

4218

In [23]:
back = spark.read.parquet("/home/jovyan/work/output/shops_long2")
back.printSchema()
back.filter(F.col("sido") == "서울특별시").count()

root
 |-- sigun: string (nullable = true)
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- menu_no: integer (nullable = true)
 |-- menu: string (nullable = true)
 |-- price_raw: string (nullable = true)
 |-- price: long (nullable = true)
 |-- sido: string (nullable = true)



4218

In [24]:
back = spark.read.parquet("/home/jovyan/work/output/shops_long2")

print("행 수:", back.count())
back.printSchema()
back.limit(10).toPandas()

행 수: 26762
root
 |-- sigun: string (nullable = true)
 |-- category: string (nullable = true)
 |-- name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- menu_no: integer (nullable = true)
 |-- menu: string (nullable = true)
 |-- price_raw: string (nullable = true)
 |-- price: long (nullable = true)
 |-- sido: string (nullable = true)



,sigun,category,name,address,menu_no,menu,price_raw,price,sido
0,수원시,한식,예전각전복설렁탕,"경기도 수원시 장안구 경수대로 816 1,2층 (영화동)",1,설렁탕,7900,7900,경기도
1,수원시,한식,예전각전복설렁탕,"경기도 수원시 장안구 경수대로 816 1,2층 (영화동)",2,선지해장국,7900,7900,경기도
2,수원시,한식,시골밥집,경기도 수원시 장안구 파장로76번길 25 (파장동),1,김치찌개,7000,7000,경기도
3,수원시,한식,풍년식당,경기도 수원시 장안구 팔달로 271번길 40 (영화동),1,설렁탕,9000,9000,경기도
4,수원시,한식,풍년식당,경기도 수원시 장안구 팔달로 271번길 40 (영화동),2,곰탕,9000,9000,경기도
5,수원시,한식,풍년식당,경기도 수원시 장안구 팔달로 271번길 40 (영화동),3,선지해장국,8000,8000,경기도
6,수원시,한식,흥부네정육식당,"경기도 수원시 장안구 정자로 146, 한솔빌딩 1층",1,삼겹살,15000,15000,경기도
7,수원시,한식,흥부네정육식당,"경기도 수원시 장안구 정자로 146, 한솔빌딩 1층",2,돼지갈비(250g),13000,13000,경기도
8,수원시,세탁업,동남세탁소,경기도 수원시 장안구 만석로101번길 46 (정자동),1,드라이클리닝,7000,7000,경기도
9,수원시,세탁업,동남세탁소,경기도 수원시 장안구 만석로101번길 46 (정자동),2,수선,4000,4000,경기도
